<a href="https://colab.research.google.com/github/HarshiniArulmani2006/Harshini-codeboosters-2026/blob/main/Day_4/Day_4_data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install pyspark --quiet
print('PySpark installation Successfully!!')

PySpark installation Successfully!!


In [33]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [34]:
#create spark session
spark=SparkSession.builder\
.appName('Day4_BigData_sales')\
.config('spark.sql.adaptive.enabled','true')\
.getOrCreate()

print(f'Spark Version: {spark.version}')
print(f'Spark Session: ACTIVE')
print(f'Application Name: {spark.sparkContext.appName}')

Spark Version: 4.0.2
Spark Session: ACTIVE
Application Name: Day4_BigData_sales


In [35]:
df_bronze=spark.read\
.option('header','true')\
.option('interSchema','true')\
.csv('large_sales_data.csv')

print('BRONZE LAYER')
print(f'Rows: {df_bronze.count()}')
print(f'Columns: {len(df_bronze.columns)}')
print(F'Names: {df_bronze.columns}')
print() #just for blank space between lines
df_bronze.printSchema()

BRONZE LAYER
Rows: 5000
Columns: 13
Names: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [36]:
print('First 5 rows: ')
df_bronze.show(5,truncate=False)
print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows: 
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi K

In [37]:
df_bronze.tail(5)

[Row(order_id='5996', customer_name='Ananya Das', product='Mouse', category='Accessories', quantity='13', unit_price='800', revenue='10400', order_date='2023-11-18', city='Bangalore', region='South', sales_rep='Meera Patel', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id='5997', customer_name='Suresh Rao', product='Webcam', category='Accessories', quantity='9', unit_price='2500', revenue='22500', order_date='2023-06-07', city='Chennai', region='South', sales_rep='Sunita Rao', payment_method='Credit Card', order_status='Delivered'),
 Row(order_id='5998', customer_name='Arjun Nair', product='Webcam', category='Accessories', quantity='1', unit_price='2500', revenue='2500', order_date='2023-04-07', city='Jaipur', region='North', sales_rep='Kavya Reddy', payment_method='Net Banking', order_status='Cancelled'),
 Row(order_id='5999', customer_name='Arjun Nair', product='Laptop', category='Electronics', quantity='14', unit_price='45000', revenue='630000', order_date='20

In [38]:
df_bronze = df_bronze \
.withColumn('unit_price', col('unit_price').cast('int')) \
.withColumn('revenue', col('revenue').cast('int'))
# Add unit_price and revenue into new column
df_bronze = df_bronze.withColumn(
    'total',
    col('unit_price') + col('revenue')
)
print('First 5 rows:')
df_bronze.select(
    'unit_price',
    'revenue',
    'total'
).show(5, truncate=False)
print('\nBasic statistics for numeric columns:')
df_bronze.select(
    'quantity',
    'unit_price',
    'revenue',
    'total'
).describe().show()

First 5 rows:
+----------+-------+------+
|unit_price|revenue|total |
+----------+-------+------+
|22000     |264000 |286000|
|12000     |120000 |132000|
|800       |8000   |8800  |
|32000     |160000 |192000|
|3500      |14000  |17500 |
+----------+-------+------+
only showing top 5 rows

Basic statistics for numeric columns:
+-------+-----------------+------------------+------------------+------------------+
|summary|         quantity|        unit_price|           revenue|             total|
+-------+-----------------+------------------+------------------+------------------+
|  count|             5000|              5000|              5000|              5000|
|   mean|           7.9536|          12496.86|          99169.52|         111666.38|
| stddev|4.275313169878912|14857.384309295603|145972.97195261103|158330.85825520795|
|    min|                1|               600|               600|              1200|
|    max|                9|             45000|            675000|           

In [39]:
df_bronze.write \
.mode('overwrite') \
.parquet('sales_bronze.parquet')
import os

print('Bronze Parquet saved: sales_bronze.parquet')

def get_dir_size(path):
    """Get total size of a file or directory in KB."""
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024
    total=0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total/1024
csv_size=get_dir_size('large_sales_data.csv')
parquet_size=get_dir_size('sales_bronze.parquet')
reduction=(1-parquet_size/csv_size)*100
print(f'\nCSV size: {csv_size:.1f} KB')
print(f'Parquet size: {parquet_size:.1f} KB')
print(f'Reduction: {reduction:.2f}% smaller')
print(f'\nAt 1 TB scale: CSV = 1000 GB --> Parquet={1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet

CSV size: 529.3 KB
Parquet size: 62.4 KB
Reduction: 88.21% smaller

At 1 TB scale: CSV = 1000 GB --> Parquet=118 GB


In [40]:
print(csv_size)
print(parquet_size)

529.3125
62.3798828125


In [41]:
df_silver=df_bronze \
   .dropDuplicates() \
   .dropna(subset=['order_id','product','revenue'])
df_silver=df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'yyyy-MM-dd')
)
df_silver=df_silver \
     .withColumn('order_year',year(col('order_date'))) \
     .withColumn('order_month',month(col('order_date')))
df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 30000, 'High')
     .when(col('revenue') > 10000, 'Medium')
     .otherwise('Low')
)
print(f'Silver layer rows:{df_silver.count()}')
print('New columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(10)

Silver layer rows:5000
New columns added: order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|   Monitor|  44000|      2023|          5|            High|
|   USB Hub|   7800|      2023|         12|             Low|
|   Printer| 120000|      2023|         10|            High|
|   Monitor| 132000|      2023|          7|            High|
|   Monitor| 220000|      2023|          5|            High|
|    Tablet|  64000|      2023|         10|            High|
|     Mouse|    800|      2023|          4|             Low|
|     Mouse|  12000|      2023|          3|          Medium|
|    Tablet| 320000|      2023|          4|            High|
|Headphones|  35000|      2023|          9|            High|
+----------+-------+----------+-----------+----------------+
only showing top 10 rows


In [42]:
#what we removed using duplicates
#duplicate products removed
df_silver=df_bronze \
   .dropDuplicates() \
   .dropna(subset=['order_id','product','revenue'])
df_silver=df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'yyyy-MM-dd')
)
df_silver=df_silver \
     .withColumn('order_year',year(col('order_date'))) \
     .withColumn('order_month',month(col('order_date')))
df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 30000, 'High')
     .when(col('revenue') > 10000, 'Medium')
     .otherwise('Low')
)
print("Duplicate products removed:")

df_bronze.groupBy("product") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

print(f'Silver layer rows:{df_silver.count()}')
print('New columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(10)

Duplicate products removed:
+----------+-----+
|product   |count|
+----------+-----+
|Speaker   |470  |
|Webcam    |532  |
|USB Hub   |527  |
|Laptop    |502  |
|Mouse     |492  |
|Tablet    |532  |
|Printer   |488  |
|Keyboard  |495  |
|Monitor   |481  |
|Headphones|481  |
+----------+-----+

Silver layer rows:5000
New columns added: order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|   Monitor|  44000|      2023|          5|            High|
|   USB Hub|   7800|      2023|         12|             Low|
|   Printer| 120000|      2023|         10|            High|
|   Monitor| 132000|      2023|          7|            High|
|   Monitor| 220000|      2023|          5|            High|
|    Tablet|  64000|      2023|         10|            High|
|     Mouse|    800|      2023|          4|             Low|
|     Mouse| 

In [43]:
df_silver.write\
.mode('overwrite')\
.parquet('sales_silver.parquet')
print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver Parquet size: {get_dir_size('sales_silver.parquet'):.1f} KB')
df_verify=spark.read.parquet('sales_silver.parquet')
print(f'Read-Back Rows:{df_verify.count()} (should match Silver Count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.parquet
Silver Parquet size: 68.8 KB
Read-Back Rows:5000 (should match Silver Count)
root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [44]:
df_silver.write\
.mode('overwrite')\
.parquet('sales_silver.parquet')
print('Silver Parquet saved: sales_silver.parquet')
print(f'Silver Parquet size: {get_dir_size('sales_silver.parquet'):.1f} KB')
print(f'Bronze Parquet size: {get_dir_size('sales_bronze.parquet'):.1f} KB')
df_verify=spark.read.parquet('sales_silver.parquet')
print(f'Read-Back Rows:{df_verify.count()} (should match Silver Count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.parquet
Silver Parquet size: 68.8 KB
Bronze Parquet size: 62.4 KB
Read-Back Rows:5000 (should match Silver Count)
root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [45]:
top_products=df_silver\
.groupBy('product')\
.agg(
    F.sum('revenue').alias('total_revenue'),
    F.count('order_id').alias('num_orders'),
    F.avg('revenue').alias('avg_order_revenue')
)\
.orderBy('total_revenue',ascending=False)\
.limit(5)
print('TOP 5 PRODUCTS BY REVENUE:')
top_products.show(truncate=False)

TOP 5 PRODUCTS BY REVENUE:
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [47]:
top_products=df_silver \
    .groupBy('product') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.round(F.avg('revenue'), 2).alias('avg_order_revenue')
    )\
    .orderBy('total_revenue',descending=False)\
    .limit(10)
print('Top 5 Products by Revenue')
top_products.show(truncate=False)

Top 5 Products by Revenue
+----------+-------------+----------+-----------------+
|product   |total_revenue|num_orders|avg_order_revenue|
+----------+-------------+----------+-----------------+
|USB Hub   |2447400      |527       |4644.02          |
|Mouse     |3207200      |492       |6518.7           |
|Keyboard  |4878000      |495       |9854.55          |
|Webcam    |10982500     |532       |20643.8          |
|Headphones|13541500     |481       |28152.81         |
|Speaker   |16317000     |470       |34717.02         |
|Printer   |44544000     |488       |91278.69         |
|Monitor   |82126000     |481       |170740.12        |
|Tablet    |135104000    |532       |253954.89        |
|Laptop    |182700000    |502       |363944.22        |
+----------+-------------+----------+-----------------+



In [50]:
top_products=df_silver \
    .groupBy('region') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.round(F.avg('revenue'), 2).alias('avg_order_revenue'),
        F.countDistinct('customer_name').alias('unique_customers'),
    )\
    .orderBy('total_revenue',ascending=False)\
    .limit(10)
print('Top 5 Products by Revenue')
top_products.show(truncate=False)

Top 5 Products by Revenue
+------+-------------+----------+-----------------+----------------+
|region|total_revenue|num_orders|avg_order_revenue|unique_customers|
+------+-------------+----------+-----------------+----------------+
|West  |198275600    |2021      |98107.67         |15              |
|South |147145900    |1483      |99221.78         |15              |
|North |99878400     |995       |100380.3         |15              |
|East  |50547700     |501       |100893.61        |15              |
+------+-------------+----------+-----------------+----------------+



In [52]:
#monthly revenue
monthly_trend=df_silver \
.groupBy('order_month')\
.agg(
    F.sum('revenue').alias('monthly_revenue'),
    F.count('revenue').alias('monthly_orders')

)\
.orderBy('order_month')
print('Monthly Revenue Trend')
monthly_trend.select('order_month','monthly_revenue','monthly_orders')
monthly_trend.show(12)


Monthly Revenue Trend
+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|          1|       41068200|           423|
|          2|       34485400|           375|
|          3|       40031200|           451|
|          4|       38857100|           390|
|          5|       39984500|           423|
|          6|       40707400|           390|
|          7|       42640700|           405|
|          8|       43718500|           418|
|          9|       37640200|           398|
|         10|       47839000|           479|
|         11|       44577100|           419|
|         12|       44298300|           429|
+-----------+---------------+--------------+

